# SPEC 004 + 005 — Test-set inference, evaluation and case analysis (Kaggle GPU)

Runs the trained nnU-Net model (SPEC 003, fold 0) **once** on the 73 locked test
cases, scores them with the official KiTS23 evaluator, and writes the case
analysis and figures.

**Before running**
1. *Add Input*: the KiTS23 images and segmentations datasets, `kits23-nnunet-preprocessed`
   (from SPEC 002, it holds `spec002/split.json`), and the training output dataset
   (the folder that holds `fold_0/checkpoint_final.pth`).
2. *Settings*: **Accelerator = GPU T4 x1** (or P100), **Internet = On**.
3. **Save Version → Save & Run All (Commit)**. Expected GPU time: roughly 1-3 hours
   (set `DISABLE_TTA = True` in step 3 to cut it to well under an hour).

**Rules this notebook follows** (docs/evaluation.md, D-012):
- post-processing is chosen on the *validation* predictions only, then applied to test;
- test results are computed once; nothing is tuned on them afterwards;
- post-processed predictions are the primary result, raw ones are reported too.

The code cells are generated by `scripts/eval/build_eval_notebook.py`.
Edit the repository, not this notebook.

In [ ]:
# Step 0 - machine check and folders
import os, shutil, sys, platform, json, time, subprocess
from pathlib import Path
print("python", sys.version.split()[0], "|", platform.platform(), "| CPUs:", os.cpu_count())
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
WORK = "/kaggle/working/project23d"
for sub in ['scripts/data', 'scripts/eval', 'src/project3d', 'src/project3d/data', 'src/project3d/eval']:
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)
T0 = time.time()

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/__init__.py
"""Project_23D: renal CT 3D segmentation."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/__init__.py
"""KiTS23 data pipeline (SPEC 002): verify, features, split, nnU-Net export."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/manifest.py
"""Locate KiTS23 files and verify them against the pinned checksum manifest.

The manifest is ``data/sources/kits23_v0.1.4_files.tsv`` (D-011): one row per
file with ``case_id, kind, size_bytes, sha256, url``. ``kind`` is ``image`` or
``segmentation``.
"""

from __future__ import annotations

import csv
import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

KINDS = ("image", "segmentation")
CASE_FILE_RE = re.compile(r"^(case_\d{5})\.nii(\.gz)?$")

# Verification statuses. Only OK is a pass; DECOMPRESSED is a warning (the file
# exists but the platform un-gzipped it, so the published hash cannot apply).
OK = "OK"
MISSING = "MISSING"
SIZE_MISMATCH = "SIZE_MISMATCH"
HASH_MISMATCH = "HASH_MISMATCH"
DECOMPRESSED = "DECOMPRESSED_UNVERIFIED"
DUPLICATE = "DUPLICATE"
FAILING = {MISSING, SIZE_MISMATCH, HASH_MISMATCH, DUPLICATE}


@dataclass(frozen=True)
class Expected:
    case_id: str
    kind: str
    size_bytes: int
    sha256: str


def load_manifest(tsv_path: str | Path) -> dict[tuple[str, str], Expected]:
    """Read the checksum manifest, keyed by ``(case_id, kind)``."""
    out: dict[tuple[str, str], Expected] = {}
    with open(tsv_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            if row["kind"] not in KINDS:
                raise ValueError(f"unknown kind {row['kind']!r} for {row['case_id']}")
            key = (row["case_id"], row["kind"])
            if key in out:
                raise ValueError(f"duplicate manifest row {key}")
            out[key] = Expected(row["case_id"], row["kind"], int(row["size_bytes"]), row["sha256"].lower())
    return out


def find_case_files(root: str | Path) -> dict[str, list[Path]]:
    """Recursively find ``case_XXXXX.nii.gz`` (or ``.nii``) files under ``root``.

    Returns every path per case so duplicates can be reported, not silently picked.
    """
    found: dict[str, list[Path]] = {}
    for path in sorted(Path(root).rglob("case_*")):
        m = CASE_FILE_RE.match(path.name)
        if m and path.is_file():
            found.setdefault(m.group(1), []).append(path)
    return found


def sha256_file(path: str | Path, chunk: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()


def verify_kind(
    manifest: dict[tuple[str, str], Expected],
    kind: str,
    root: str | Path,
    check_hash: bool = True,
) -> list[dict]:
    """Check every manifest entry of one kind against the files under ``root``.

    Returns one report row per manifest entry, plus one row per file found
    under ``root`` that the manifest does not list (status ``UNEXPECTED``).
    """
    found = find_case_files(root)
    rows = []
    for (case_id, k), exp in sorted(manifest.items()):
        if k != kind:
            continue
        paths = found.pop(case_id, [])
        row = {"case_id": case_id, "kind": kind, "path": "", "status": OK,
               "size_bytes": "", "sha256": ""}
        if not paths:
            row["status"] = MISSING
        elif len(paths) > 1:
            row.update(status=DUPLICATE, path=";".join(str(p) for p in paths))
        else:
            path = paths[0]
            row["path"] = str(path)
            row["size_bytes"] = path.stat().st_size
            if path.suffix != ".gz":
                row["status"] = DECOMPRESSED
            elif row["size_bytes"] != exp.size_bytes:
                row["status"] = SIZE_MISMATCH
            elif check_hash:
                row["sha256"] = sha256_file(path)
                if row["sha256"] != exp.sha256:
                    row["status"] = HASH_MISMATCH
        rows.append(row)
    for case_id, paths in sorted(found.items()):
        for p in paths:
            rows.append({"case_id": case_id, "kind": kind, "path": str(p),
                         "status": "UNEXPECTED", "size_bytes": p.stat().st_size, "sha256": ""})
    return rows


def summarize(rows: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for r in rows:
        counts[r["status"]] = counts.get(r["status"], 0) + 1
    return counts


def resolved_paths(rows: list[dict]) -> dict[str, Path]:
    """Map case_id to its single verified path (OK or DECOMPRESSED rows only)."""
    return {r["case_id"]: Path(r["path"]) for r in rows if r["status"] in (OK, DECOMPRESSED)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/features.py
"""Per-case label features used to stratify the split (D-012).

Only segmentations are read. Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

VALID_LABELS = {0, 1, 2, 3}
KITS21_LAST_ID = 299  # cases 0-299 are the KiTS21 cohort; 400+ were added in KiTS23
SIZE_BINS = ("none", "small", "medium", "large")


def cohort_of(case_id: str) -> str:
    """Cohort proxy for contrast phase (D-010, D-012); not a phase label."""
    return "kits21" if int(case_id.split("_")[1]) <= KITS21_LAST_ID else "kits23_new"


def label_features(seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """Volumes in millilitres and presence flags from a label array."""
    present = {int(v) for v in np.unique(seg)}
    unknown = present - VALID_LABELS
    if unknown:
        raise ValueError(f"unexpected label values {sorted(unknown)}")
    counts = np.bincount(seg.ravel().astype(np.int64), minlength=4)
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    return {
        "kidney_ml": round(counts[1] * voxel_ml, 3),
        "tumor_ml": round(counts[2] * voxel_ml, 3),
        "cyst_ml": round(counts[3] * voxel_ml, 3),
        "has_tumor": bool(counts[2] > 0),
        "has_cyst": bool(counts[3] > 0),
    }


def case_features(case_id: str, seg_path: str | Path) -> dict:
    """Read one segmentation and return its features plus header geometry."""
    import nibabel as nib

    img = nib.load(str(seg_path))
    seg = np.asanyarray(img.dataobj)
    if not np.all(np.mod(seg, 1) == 0):
        raise ValueError(f"{case_id}: non-integer label values")
    seg = seg.astype(np.uint8)
    spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
    row = {"case_id": case_id, "cohort": cohort_of(case_id),
           "shape": "x".join(map(str, seg.shape)),
           "spacing_mm": "x".join(f"{s:.4f}" for s in spacing)}
    row.update(label_features(seg, spacing))
    return row


def geometry_mismatch(image_path: str | Path, seg_path: str | Path, atol: float = 1e-3) -> str | None:
    """Compare image and mask headers (no voxel data read). None if they match."""
    import nibabel as nib

    img, seg = nib.load(str(image_path)), nib.load(str(seg_path))
    if img.shape[:3] != seg.shape[:3]:
        return f"shape {img.shape[:3]} vs {seg.shape[:3]}"
    if not np.allclose(img.affine, seg.affine, atol=atol):
        return "affine differs"
    return None


def add_size_bins(rows: list[dict]) -> tuple[float, float]:
    """Add ``size_bin`` in place: tertiles of tumor volume among tumor cases.

    Cases without tumor get ``none``. Returns the two cut points (ml) so they
    can be recorded with the split.
    """
    vols = np.array([r["tumor_ml"] for r in rows if r["has_tumor"]], dtype=float)
    if vols.size == 0:
        cuts = (0.0, 0.0)
    else:
        q1, q2 = np.quantile(vols, [1 / 3, 2 / 3])
        cuts = (float(q1), float(q2))
    for r in rows:
        if not r["has_tumor"]:
            r["size_bin"] = "none"
        elif r["tumor_ml"] <= cuts[0]:
            r["size_bin"] = "small"
        elif r["tumor_ml"] <= cuts[1]:
            r["size_bin"] = "medium"
        else:
            r["size_bin"] = "large"
    return cuts


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/split.py
"""Case-level stratified train/validation/test split (SPEC 002, D-012).

Strata are ``cohort | size_bin | cyst``. Per-stratum counts are allocated by
largest-remainder apportionment, so the global counts hit the target fractions
exactly and every stratum is represented proportionally. Cases are shuffled
within each stratum with a fixed seed.

The saved ``split.json`` is the authority. Its fingerprint is recorded so any
later change is detectable; the split is never regenerated after training.
"""

from __future__ import annotations

import hashlib
import json
import math
import random

ROLES = ("train", "val", "test")


def stratum_key(row: dict) -> str:
    return f"{row['cohort']}|{row['size_bin']}|{'cyst' if row['has_cyst'] else 'nocyst'}"


def _apportion(sizes: dict[str, int], total: int, capacity: dict[str, int]) -> dict[str, int]:
    """Split ``total`` across strata proportionally to ``sizes``, within ``capacity``."""
    n = sum(sizes.values())
    if n == 0 or total == 0:
        return {k: 0 for k in sizes}
    quota = {k: sizes[k] * total / n for k in sizes}
    alloc = {k: min(math.floor(quota[k]), capacity[k]) for k in sizes}
    remaining = total - sum(alloc.values())
    # Largest remainder first; ties broken by stratum name for determinism.
    order = sorted(sizes, key=lambda k: (-(quota[k] - math.floor(quota[k])), k))
    while remaining > 0:
        progressed = False
        for k in order:
            if remaining == 0:
                break
            if alloc[k] < capacity[k]:
                alloc[k] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            raise ValueError("not enough cases to satisfy the requested split")
    return alloc


def make_split(rows: list[dict], seed: int = 42,
               fractions: tuple[float, float, float] = (0.70, 0.15, 0.15)) -> dict[str, str]:
    """Return ``{case_id: role}`` with role in ``train``/``val``/``test``."""
    if abs(sum(fractions) - 1.0) > 1e-9:
        raise ValueError("fractions must sum to 1")
    ids = [r["case_id"] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError("duplicate case_id in feature rows")

    strata: dict[str, list[str]] = {}
    for r in rows:
        strata.setdefault(stratum_key(r), []).append(r["case_id"])
    sizes = {k: len(v) for k, v in strata.items()}
    n = len(rows)
    n_test = round(n * fractions[2])
    n_val = round(n * fractions[1])

    test_alloc = _apportion(sizes, n_test, sizes)
    val_alloc = _apportion(sizes, n_val, {k: sizes[k] - test_alloc[k] for k in sizes})

    rng = random.Random(seed)
    roles: dict[str, str] = {}
    for key in sorted(strata):
        members = sorted(strata[key])
        rng.shuffle(members)
        t, v = test_alloc[key], val_alloc[key]
        for i, cid in enumerate(members):
            roles[cid] = "test" if i < t else "val" if i < t + v else "train"
    return dict(sorted(roles.items()))


def fingerprint(roles: dict[str, str]) -> str:
    """SHA-256 of the canonical ``{case_id: role}`` mapping."""
    canon = json.dumps(dict(sorted(roles.items())), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def nnunet_splits(roles: dict[str, str]) -> list[dict[str, list[str]]]:
    """nnU-Net ``splits_final.json`` content: one fold, train vs val. Test is excluded."""
    return [{
        "train": sorted(c for c, r in roles.items() if r == "train"),
        "val": sorted(c for c, r in roles.items() if r == "val"),
    }]


def summary(rows: list[dict], roles: dict[str, str]) -> dict:
    """Counts per role, overall and by each stratification variable."""
    out: dict = {"total": {r: 0 for r in ROLES}}
    for var in ("cohort", "size_bin", "has_cyst"):
        out[var] = {}
    for row in rows:
        role = roles[row["case_id"]]
        out["total"][role] += 1
        for var in ("cohort", "size_bin", "has_cyst"):
            bucket = out[var].setdefault(str(row[var]), {r: 0 for r in ROLES})
            bucket[role] += 1
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/nnunet_export.py
"""Build the nnU-Net v2 raw dataset folder for KiTS23 (SPEC 002).

Mirrors nnU-Net's own ``dataset_conversion/Dataset220_KiTS2023.py``: region-based
labels (kidney+masses, masses, tumor), ``regions_class_order`` (1, 3, 2) and the
``NibabelIOWithReorient`` reader. Two differences, both deliberate:

- only train and validation cases go into ``imagesTr``/``labelsTr``, so the
  locked test cases cannot influence nnU-Net's planning or normalization;
- files are symlinked by default instead of copied, so no extra disk is used.
"""

from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

DATASET_ID = 220
DATASET_NAME = "KiTS2023"
DATASET_JSON = {
    "channel_names": {"0": "CT"},
    "labels": {"background": 0, "kidney": [1, 2, 3], "masses": [2, 3], "tumor": 2},
    "regions_class_order": [1, 3, 2],
    "overwrite_image_reader_writer": "NibabelIOWithReorient",
    "name": DATASET_NAME,
    "release": "0.1.4",
    "reference": "https://github.com/neheller/kits23",
    "licence": "CC BY-NC-SA 4.0 (KiTS23 data terms)",
    "description": "KiTS23 train+val cases only; locked test cases excluded (D-012)",
}


def dataset_folder_name(dataset_id: int = DATASET_ID, name: str = DATASET_NAME) -> str:
    return f"Dataset{dataset_id:03d}_{name}"


def _file_ending(paths: list[Path]) -> str:
    endings = {".nii.gz" if p.name.endswith(".nii.gz") else p.suffix for p in paths}
    if len(endings) != 1:
        raise ValueError(f"mixed file endings {sorted(endings)}; nnU-Net needs one")
    return endings.pop()


def _place(src: Path, dst: Path, mode: str) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if mode == "symlink":
        os.symlink(src.resolve(), dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError(f"unknown mode {mode!r}")


def build_raw_dataset(roles: dict[str, str], images: dict[str, Path], segs: dict[str, Path],
                      raw_root: str | Path, mode: str = "symlink",
                      dataset_id: int = DATASET_ID) -> Path:
    """Create ``<raw_root>/DatasetXXX_KiTS2023`` and return its path.

    ``imagesTr``/``labelsTr`` hold train+val cases; ``imagesTs`` holds test images.
    Test labels are never placed in the nnU-Net folder.
    """
    trainval = sorted(c for c, r in roles.items() if r in ("train", "val"))
    test = sorted(c for c, r in roles.items() if r == "test")
    missing = [c for c in roles if c not in images or (c in trainval and c not in segs)]
    if missing:
        raise ValueError(f"no verified file for {len(missing)} cases, e.g. {missing[:3]}")

    ending = _file_ending([images[c] for c in roles] + [segs[c] for c in trainval])
    out = Path(raw_root) / dataset_folder_name(dataset_id)
    if out.exists():
        shutil.rmtree(out)
    for c in trainval:
        _place(images[c], out / "imagesTr" / f"{c}_0000{ending}", mode)
        _place(segs[c], out / "labelsTr" / f"{c}{ending}", mode)
    for c in test:
        _place(images[c], out / "imagesTs" / f"{c}_0000{ending}", mode)

    meta = dict(DATASET_JSON, numTraining=len(trainval), file_ending=ending)
    (out / "dataset.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return out


def check_no_test_leak(dataset_dir: str | Path, roles: dict[str, str]) -> None:
    """Fail if any test case appears in imagesTr or labelsTr."""
    test = {c for c, r in roles.items() if r == "test"}
    for sub in ("imagesTr", "labelsTr"):
        for p in (Path(dataset_dir) / sub).iterdir():
            if p.name[:10] in test:
                raise AssertionError(f"test case {p.name[:10]} found in {sub}")


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/__init__.py
"""SPEC 004-005: test-set metrics, statistics and case analysis."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/metrics.py
"""Secondary per-case metrics (SPEC 004).

The co-primary Dice and surface Dice come from the official KiTS23 evaluator
(see ``official.py``). This module adds the secondary metrics from
docs/evaluation.md, with the official empty-mask rule for Dice and IoU:
both empty -> 1, exactly one empty -> 0. Precision, recall and HD95 are
undefined (NaN) when their denominator mask is empty; counts are reported.
"""

from __future__ import annotations

import numpy as np
from scipy import ndimage

# Official KiTS hierarchical evaluation classes first, then the remaining single classes.
HECS = {"kidney_and_mass": (1, 2, 3), "mass": (2, 3), "tumor": (2,)}
REGIONS = {**HECS, "kidney": (1,), "cyst": (3,)}
# Copied from kits23.configuration.labels; official.check_config() asserts they still match.
SD_TOLERANCES_MM = {"kidney_and_mass": 1.0330772532390826, "mass": 1.1328796488598762,
                    "tumor": 1.1498198361434828}


def region_mask(seg: np.ndarray, labels: tuple[int, ...]) -> np.ndarray:
    return np.isin(seg, labels)


def overlap(pred: np.ndarray, ref: np.ndarray) -> dict:
    """Dice, IoU, precision and recall for two boolean masks."""
    tp = int(np.count_nonzero(pred & ref))
    n_pred, n_ref = int(np.count_nonzero(pred)), int(np.count_nonzero(ref))
    if n_pred == 0 and n_ref == 0:
        dice = iou = 1.0
    elif n_pred == 0 or n_ref == 0:
        dice = iou = 0.0
    else:
        dice = 2 * tp / (n_pred + n_ref)
        iou = tp / (n_pred + n_ref - tp)
    return {"dice": dice, "iou": iou,
            "precision": tp / n_pred if n_pred else float("nan"),
            "recall": tp / n_ref if n_ref else float("nan")}


def _surface(mask: np.ndarray) -> np.ndarray:
    return mask & ~ndimage.binary_erosion(mask, border_value=0)


def hd95(pred: np.ndarray, ref: np.ndarray, spacing_mm: tuple[float, ...]) -> float:
    """95th-percentile symmetric Hausdorff distance in mm (max of the two directions).

    NaN when either mask is empty. Computed inside the bounding box of both
    masks, which holds every surface voxel, so distances are unchanged.
    """
    if not pred.any() or not ref.any():
        return float("nan")
    idx = np.argwhere(pred | ref)
    lo, hi = np.maximum(idx.min(0) - 1, 0), idx.max(0) + 2
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    p = np.pad(pred[box], 1)
    r = np.pad(ref[box], 1)
    sp, sr = _surface(p), _surface(r)
    dist_to_r = ndimage.distance_transform_edt(~sr, sampling=spacing_mm)
    dist_to_p = ndimage.distance_transform_edt(~sp, sampling=spacing_mm)
    return float(max(np.percentile(dist_to_r[sp], 95), np.percentile(dist_to_p[sr], 95)))


def case_metrics(pred_seg: np.ndarray, ref_seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """All secondary metrics for one case, flattened as ``<region>_<metric>``."""
    if pred_seg.shape != ref_seg.shape:
        raise ValueError(f"shape {pred_seg.shape} vs {ref_seg.shape}")
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    out: dict = {}
    for name, labels in REGIONS.items():
        p, r = region_mask(pred_seg, labels), region_mask(ref_seg, labels)
        for k, v in overlap(p, r).items():
            out[f"{name}_{k}"] = v
        out[f"{name}_hd95_mm"] = hd95(p, r, spacing_mm)
        out[f"{name}_ref_ml"] = round(float(np.count_nonzero(r)) * voxel_ml, 3)
        out[f"{name}_pred_ml"] = round(float(np.count_nonzero(p)) * voxel_ml, 3)
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/stats.py
"""Per-case distribution summaries and bootstrap confidence intervals."""

from __future__ import annotations

import numpy as np

N_BOOT = 10_000
MIN_CASES_FOR_CI = 10  # below this a CI is not meaningful; reported as None


def describe(values) -> dict:
    """n, mean, median, IQR, 10th percentile, min and max, ignoring NaN (counted)."""
    a = np.asarray(values, dtype=float)
    ok = a[~np.isnan(a)]
    out = {"n": int(ok.size), "n_undefined": int(a.size - ok.size)}
    if ok.size == 0:
        return out
    q10, q1, med, q3 = np.percentile(ok, [10, 25, 50, 75])
    out.update(mean=float(ok.mean()), median=float(med), q1=float(q1), q3=float(q3),
               p10=float(q10), min=float(ok.min()), max=float(ok.max()))
    return out


def bootstrap_ci(values, n_boot: int = N_BOOT, seed: int = 42, alpha: float = 0.05):
    """Percentile bootstrap CI of the mean over cases; None if too few cases."""
    a = np.asarray(values, dtype=float)
    a = a[~np.isnan(a)]
    if a.size < MIN_CASES_FOR_CI:
        return None
    rng = np.random.default_rng(seed)
    means = a[rng.integers(0, a.size, size=(n_boot, a.size))].mean(axis=1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return [float(lo), float(hi)]


def describe_with_ci(values, **kw) -> dict:
    return {**describe(values), "ci95": bootstrap_ci(values, **kw)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/official.py
"""Thin wrapper around the official KiTS23 evaluator (co-primary metrics, D-012).

Requires ``pip install git+https://github.com/neheller/kits23`` (done in the
Kaggle notebook). The official ``compute_metrics_for_case`` returns a 3x2
array: HECs (kidney_and_mass, mass, tumor) x (Dice, surface Dice). Its own
shape/spacing checks are ineffective, so evaluate.py checks geometry first.
"""

from __future__ import annotations

from .metrics import HECS, SD_TOLERANCES_MM


def check_config() -> None:
    """Fail if the installed evaluator's classes or tolerances differ from ours."""
    from kits23.configuration import labels

    assert list(labels.HEC_NAME_LIST) == list(HECS), labels.HEC_NAME_LIST
    for name, lab in HECS.items():
        assert tuple(labels.KITS_HEC_LABEL_MAPPING[name]) == lab, name
        assert abs(labels.HEC_SD_TOLERANCES_MM[name] - SD_TOLERANCES_MM[name]) < 1e-12, name


def official_case(pred_path: str, ref_path: str) -> dict:
    from kits23.evaluation.metrics import compute_metrics_for_case

    m = compute_metrics_for_case(str(pred_path), str(ref_path))
    out = {}
    for i, name in enumerate(HECS):
        out[f"{name}_dice_official"] = float(m[i, 0])
        out[f"{name}_sd_official"] = float(m[i, 1])
    return out


def official_version() -> str:
    """Installed kits23 version plus the git commit pip recorded, if any."""
    import importlib.metadata as md
    import json

    dist = md.distribution("kits23")
    commit = ""
    direct = dist.read_text("direct_url.json")
    if direct:
        commit = json.loads(direct).get("vcs_info", {}).get("commit_id", "")
    return f"{dist.version} {commit}".strip()


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/analysis.py
"""SPEC 005 / E-005: performance by case characteristics (no GPU).

Descriptive only: group sizes are small (73 test cases), so the tables give
n, mean, median and a bootstrap CI per group, and no significance claims are
made (docs/evaluation.md, statistical reporting).
"""

from __future__ import annotations

import numpy as np
from scipy import stats as sps

from .stats import bootstrap_ci, describe

SIZE_ORDER = ["none", "small", "medium", "large"]
METRICS = ["mean_dice", "mean_sd", "kidney_and_mass_dice", "mass_dice", "tumor_dice"]
FACTORS = ["size_bin", "has_cyst", "cohort", "fg_volume_quartile"]


def _num(v) -> float:
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")


def _truthy(v) -> bool:
    return str(v).strip().lower() in ("true", "1")


def join_rows(per_case: list[dict], case_features: list[dict]) -> list[dict]:
    """Merge metrics with SPEC 002 features and add the foreground-volume quartile.

    ``tumor_dice`` etc. prefer the official columns when present.
    """
    feats = {r["case_id"]: r for r in case_features}
    rows = []
    for pc in per_case:
        f = feats[pc["case_id"]]
        row = {"case_id": pc["case_id"], "cohort": f["cohort"], "size_bin": f["size_bin"],
               "has_cyst": _truthy(f["has_cyst"]), "has_tumor": _truthy(f["has_tumor"]),
               "tumor_ml": _num(f["tumor_ml"]),
               "fg_ml": _num(f["kidney_ml"]) + _num(f["tumor_ml"]) + _num(f["cyst_ml"]),
               "mean_dice": _num(pc["mean_dice"]), "mean_sd": _num(pc["mean_sd"]),
               "tumor_pred_ml": _num(pc["tumor_pred_ml"])}
        for h in ("kidney_and_mass", "mass", "tumor"):
            row[f"{h}_dice"] = _num(pc.get(f"{h}_dice_official", pc[f"{h}_dice"]))
        rows.append(row)
    fg = np.array([r["fg_ml"] for r in rows])
    cuts = np.quantile(fg, [0.25, 0.5, 0.75]) if len(rows) else []
    for r in rows:
        r["fg_volume_quartile"] = f"Q{int(np.searchsorted(cuts, r['fg_ml'], side='left')) + 1}"
    return rows


def group_table(rows: list[dict], factor: str, metric: str) -> list[dict]:
    groups = sorted({r[factor] for r in rows},
                    key=lambda g: SIZE_ORDER.index(g) if g in SIZE_ORDER else str(g))
    out = []
    for g in groups:
        vals = [r[metric] for r in rows if r[factor] == g]
        d = describe(vals)
        out.append({"group": str(g), "n": d["n"], "mean": d.get("mean"),
                    "median": d.get("median"), "ci95": bootstrap_ci(vals)})
    return out


def spearman_with_ci(x, y, n_boot: int = 2000, seed: int = 42) -> dict:
    """Spearman correlation and a case-bootstrap 95% CI (None if under 10 pairs)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if x.size < 10:
        return {"n": int(x.size), "rho": None, "ci95": None}
    rho = float(sps.spearmanr(x, y).statistic)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, x.size, x.size)
        r = sps.spearmanr(x[i], y[i]).statistic
        if not np.isnan(r):
            boots.append(r)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"n": int(x.size), "rho": rho, "ci95": [float(lo), float(hi)]}


def analyse(rows: list[dict], n_worst: int = 10) -> dict:
    tables = {f: {m: group_table(rows, f, m) for m in METRICS} for f in FACTORS}
    tumor = [r for r in rows if r["has_tumor"]]
    no_tumor = [r for r in rows if not r["has_tumor"]]
    worst = sorted(rows, key=lambda r: r["mean_dice"])[:n_worst]
    return {
        "n_cases": len(rows),
        "tables": tables,
        "tumor_volume_vs_tumor_dice": spearman_with_ci(
            [r["tumor_ml"] for r in tumor], [r["tumor_dice"] for r in tumor]),
        "no_tumor_cases": {"n": len(no_tumor),
                           "n_with_predicted_tumor": sum(r["tumor_pred_ml"] > 0 for r in no_tumor)},
        "worst_cases": [{k: r[k] for k in ("case_id", "mean_dice", "tumor_dice", "size_bin",
                                           "has_cyst", "cohort", "tumor_ml")} for r in worst],
    }


def pick_examples(rows: list[dict], metric: str = "mean_dice") -> dict:
    """Best, median and worst case by ``metric`` (ties broken by case ID)."""
    s = sorted(rows, key=lambda r: (r[metric], r["case_id"]))
    return {"worst": s[0]["case_id"], "median": s[len(s) // 2]["case_id"], "best": s[-1]["case_id"]}


def to_markdown(result: dict) -> str:
    def fmt(v):
        return "–" if v is None else f"{v:.3f}"

    def ci(c):
        return "–" if c is None else f"{c[0]:.3f} to {c[1]:.3f}"

    lines = [f"# E-005 — Performance by case characteristics (n = {result['n_cases']})", "",
             "Descriptive only; groups are small and no significance test is made.", ""]
    for factor, by_metric in result["tables"].items():
        lines += [f"## {factor}", "",
                  "| group | n | mean Dice | 95% CI | tumor Dice | 95% CI | mean surface Dice |",
                  "|---|---|---|---|---|---|---|"]
        for md, td, sd in zip(by_metric["mean_dice"], by_metric["tumor_dice"], by_metric["mean_sd"]):
            lines.append(f"| {md['group']} | {md['n']} | {fmt(md['mean'])} | {ci(md['ci95'])} | "
                         f"{fmt(td['mean'])} | {ci(td['ci95'])} | {fmt(sd['mean'])} |")
        lines.append("")
    sp = result["tumor_volume_vs_tumor_dice"]
    lines += ["## Tumor volume vs tumor Dice (tumor cases)", "",
              f"Spearman rho = {fmt(sp['rho'])}, 95% CI {ci(sp['ci95'])}, n = {sp['n']}.", ""]
    nt = result["no_tumor_cases"]
    lines += ["## Cases without tumor", "",
              f"{nt['n_with_predicted_tumor']} of {nt['n']} predicted some tumor (false positive).", "",
              "## Worst cases by mean Dice", "", "| case | mean Dice | tumor Dice | size bin | cyst | cohort |",
              "|---|---|---|---|---|---|"]
    for w in result["worst_cases"]:
        lines.append(f"| {w['case_id']} | {w['mean_dice']:.3f} | {w['tumor_dice']:.3f} | "
                     f"{w['size_bin']} | {w['has_cyst']} | {w['cohort']} |")
    return "\n".join(lines) + "\n"


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/figures.py
"""Qualitative figures for SPEC 005: orthogonal overlays and a box plot.

Reference contours are drawn solid, predictions dashed. CT is shown with a
soft-tissue window (level 40, width 400 HU) for display only.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

WINDOW = (-160.0, 240.0)
COLORS = {"kidney_and_mass": "#2ca02c", "mass": "#ff7f0e", "tumor": "#d62728"}
LABELS = {"kidney_and_mass": (1, 2, 3), "mass": (2, 3), "tumor": (2,)}


def focus_index(ref: np.ndarray) -> tuple[int, int, int]:
    """Voxel to centre the views on: largest mass slice, else kidney, else the middle."""
    for labels in ((2, 3), (1, 2, 3)):
        m = np.isin(ref, labels)
        if m.any():
            k = int(np.argmax(m.sum(axis=(1, 2))))
            j = int(np.argmax(m.sum(axis=(0, 2))))
            i = int(np.argmax(m.sum(axis=(0, 1))))
            return k, j, i
    return tuple(s // 2 for s in ref.shape)


def overlay_figure(image: np.ndarray, ref: np.ndarray, pred: np.ndarray,
                   spacing_mm: tuple[float, float, float], title: str, out_path: Path) -> Path:
    """Axial, coronal and sagittal views through the focus voxel.

    Arrays are in file order (axis 0 = slices, as in KiTS23 NIfTI files).
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    k, j, i = focus_index(ref)
    s0, s1, s2 = spacing_mm
    # (name, slice, pixel aspect, image origin): "upper" puts the spine at the bottom
    # of the axial view, as radiologists read it.
    views = [("axial", np.s_[k, :, :], s1 / s2, "upper"), ("coronal", np.s_[:, j, :], s0 / s2, "lower"),
             ("sagittal", np.s_[:, :, i], s0 / s1, "lower")]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
    for ax, (name, sl, aspect, origin) in zip(axes, views):
        ax.imshow(np.clip(image[sl], *WINDOW), cmap="gray", aspect=aspect, origin=origin)
        for region, labels in LABELS.items():
            for arr, style in ((ref, "solid"), (pred, "dashed")):
                m = np.isin(arr[sl], labels).astype(float)
                if m.any():
                    ax.contour(m, levels=[0.5], colors=COLORS[region], linestyles=style,
                               linewidths=1, origin=origin)
        ax.set_title(name)
        ax.axis("off")
    fig.suptitle(f"{title}   (solid = reference, dashed = prediction; "
                 f"green kidney+mass, orange mass, red tumor)", fontsize=9)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    return out_path


def boxplot_by_group(rows: list[dict], factor: str, metric: str, order: list[str],
                     out_path: Path) -> Path:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    groups = [g for g in order if any(str(r[factor]) == g for r in rows)]
    data = [[r[metric] for r in rows if str(r[factor]) == g] for g in groups]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(groups) + 1), [f"{g}\n(n={len(d)})" for g, d in zip(groups, data)])
    ax.set_ylabel(metric)
    ax.set_xlabel(factor)
    ax.set_ylim(-0.02, 1.02)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    return out_path


In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/prepare_kits23.py
"""SPEC 002 pipeline: verify -> features -> split -> nnU-Net raw dataset.

Runs on a Kaggle CPU session (or locally on a small subset). Example::

    python scripts/data/prepare_kits23.py --input-root /kaggle/input \
        --manifest data/sources/kits23_v0.1.4_files.tsv \
        --out-dir /kaggle/working/spec002 --raw-root /tmp/nnUNet_raw

Outputs in ``--out-dir``: verify_report.csv, case_features.csv, split.json,
splits_final.json. The nnU-Net raw dataset is written under ``--raw-root``.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import statistics
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest, nnunet_export, split  # noqa: E402

IMAGE_MIN_BYTES = 10 * 1024 * 1024  # CT volumes are tens of MB; label files are ~1 MB


def detect_roots(input_root: Path, min_files: int = 1) -> tuple[Path, Path]:
    """Find the images folder and the segmentations folder by typical file size.

    Folders with fewer than ``min_files`` case files are ignored (e.g. nnU-Net's
    validation predictions sitting next to the full dataset).
    """
    by_dir: dict[Path, list[int]] = {}
    for paths in manifest.find_case_files(input_root).values():
        for p in paths:
            by_dir.setdefault(p.parent, []).append(p.stat().st_size)
    by_dir = {d: s for d, s in by_dir.items() if len(s) >= min_files}
    if not by_dir:
        raise SystemExit(f"no case_XXXXX.nii(.gz) files found under {input_root}")
    big = [d for d, s in by_dir.items() if statistics.median(s) >= IMAGE_MIN_BYTES]
    small = [d for d, s in by_dir.items() if statistics.median(s) < IMAGE_MIN_BYTES]
    if len(big) != 1 or len(small) != 1:
        raise SystemExit(f"could not tell images from segmentations; folders found: "
                         f"{ {str(d): len(s) for d, s in by_dir.items()} }. "
                         f"Pass --images-root and --segs-root explicitly.")
    return big[0], small[0]


def write_csv(path: Path, rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--manifest", type=Path, required=True)
    ap.add_argument("--input-root", type=Path, help="auto-detect images/segmentation folders here")
    ap.add_argument("--images-root", type=Path)
    ap.add_argument("--segs-root", type=Path)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--raw-root", type=Path, required=True)
    ap.add_argument("--link-mode", choices=("symlink", "copy"), default="symlink")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--subset", action="store_true",
                    help="allow missing cases (local smoke test); never use for the real run")
    ap.add_argument("--skip-hash", action="store_true", help="size check only (faster, weaker)")
    args = ap.parse_args(argv)

    t0 = time.time()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    if args.images_root and args.segs_root:
        images_root, segs_root = args.images_root, args.segs_root
    elif args.input_root:
        images_root, segs_root = detect_roots(args.input_root)
    else:
        ap.error("give --input-root, or both --images-root and --segs-root")
    print(f"images: {images_root}\nsegmentations: {segs_root}")

    # 1. Verify every file against the pinned manifest.
    exp = manifest.load_manifest(args.manifest)
    report = (manifest.verify_kind(exp, "image", images_root, not args.skip_hash)
              + manifest.verify_kind(exp, "segmentation", segs_root, not args.skip_hash))
    write_csv(args.out_dir / "verify_report.csv", report)
    counts = manifest.summarize(report)
    print(f"[1/4] verify: {counts}  ({time.time() - t0:.0f}s)")
    failing = {s: n for s, n in counts.items() if s in manifest.FAILING or s == "UNEXPECTED"}
    if args.subset:
        failing.pop(manifest.MISSING, None)
    if failing:
        print("STOP: verification failed; see verify_report.csv")
        return 1
    if counts.get(manifest.DECOMPRESSED):
        print("WARNING: some files were un-gzipped by the platform; their hashes cannot be checked")

    images = manifest.resolved_paths([r for r in report if r["kind"] == "image"])
    segs = manifest.resolved_paths([r for r in report if r["kind"] == "segmentation"])
    cases = sorted(set(images) & set(segs))
    if not args.subset and len(cases) != len({c for c, _ in exp}):
        print(f"STOP: only {len(cases)} cases have both image and label")
        return 1

    # 2. Image/mask geometry must agree (SPEC 001 5.4), then label features per case.
    bad = {c: why for c in cases if (why := features.geometry_mismatch(images[c], segs[c]))}
    if bad:
        print(f"STOP: image/mask geometry mismatch: {bad}")
        return 1
    rows = [features.case_features(c, segs[c]) for c in cases]
    cuts = features.add_size_bins(rows)
    write_csv(args.out_dir / "case_features.csv", rows)
    print(f"[2/4] features: {len(rows)} cases, tumor-size tertile cuts (ml) = "
          f"{cuts[0]:.2f}, {cuts[1]:.2f}  ({time.time() - t0:.0f}s)")

    # 3. Split once, fingerprint, save.
    roles = split.make_split(rows, seed=args.seed)
    record = {
        "spec": "SPEC 002",
        "dataset": "KiTS23 v0.1.4",
        # Line endings normalized so Windows and Kaggle copies hash the same.
        "manifest_sha256": hashlib.sha256(
            args.manifest.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
        "seed": args.seed,
        "fractions": {"train": 0.70, "val": 0.15, "test": 0.15},
        "strata": "cohort | size_bin | cyst",
        "size_bin_cuts_ml": list(cuts),
        "fingerprint_sha256": split.fingerprint(roles),
        "summary": split.summary(rows, roles),
        "roles": roles,
    }
    (args.out_dir / "split.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    (args.out_dir / "splits_final.json").write_text(
        json.dumps(split.nnunet_splits(roles), indent=2), encoding="utf-8")
    print(f"[3/4] split: {record['summary']['total']}  fingerprint {record['fingerprint_sha256'][:16]}...")

    # 4. nnU-Net raw dataset (train+val only in imagesTr).
    ds = nnunet_export.build_raw_dataset(roles, images, segs, args.raw_root, mode=args.link_mode)
    nnunet_export.check_no_test_leak(ds, roles)
    print(f"[4/4] nnU-Net raw dataset: {ds}  ({time.time() - t0:.0f}s total)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/eval/evaluate.py
"""SPEC 004: score predictions on the locked test set.

Example (Kaggle)::

    python scripts/eval/evaluate.py --pred-dir /kaggle/working/spec004/pred_pp \
        --segs-root <KiTS23 segmentations> --split split.json --role test \
        --out-dir /kaggle/working/spec004/eval

Writes per_case.csv and summary.json. Co-primary metrics are the official KiTS23
Dice and surface Dice, averaged over the three HECs per case, then over cases
(D-012). ``--no-official`` is for local smoke tests only: the summary is then
marked ``primary_valid: false``.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest  # noqa: E402
from project3d.eval import metrics, stats  # noqa: E402

CROSSCHECK_TOL = 1e-3  # our Dice vs official Dice; larger means a geometry/orientation bug


def score_case(args: tuple) -> dict:
    case_id, pred_path, ref_path, use_official = args
    import nibabel as nib

    why = features.geometry_mismatch(pred_path, ref_path)
    if why:
        raise ValueError(f"{case_id}: prediction/reference geometry mismatch ({why})")
    ref_img = nib.load(str(ref_path))
    ref = np.asanyarray(ref_img.dataobj).astype(np.uint8)
    pred = np.asanyarray(nib.load(str(pred_path)).dataobj).astype(np.uint8)
    bad = set(np.unique(pred).tolist()) - features.VALID_LABELS
    if bad:
        raise ValueError(f"{case_id}: prediction has labels {sorted(bad)}")
    spacing = tuple(float(z) for z in ref_img.header.get_zooms()[:3])
    row = {"case_id": case_id, **metrics.case_metrics(pred, ref, spacing)}
    if use_official:
        from project3d.eval import official

        row.update(official.official_case(pred_path, ref_path))
        row["mean_dice"] = float(np.mean([row[f"{h}_dice_official"] for h in metrics.HECS]))
        row["mean_sd"] = float(np.mean([row[f"{h}_sd_official"] for h in metrics.HECS]))
    else:
        row["mean_dice"] = float(np.mean([row[f"{h}_dice"] for h in metrics.HECS]))
        row["mean_sd"] = float("nan")
    return row


def summarize(rows: list[dict], official: bool) -> dict:
    col = lambda k: [r[k] for r in rows]  # noqa: E731
    out = {
        "n_cases": len(rows),
        "official_evaluator": official,
        "primary_valid": official,
        "primary": {"mean_dice": stats.describe_with_ci(col("mean_dice")),
                    "mean_surface_dice": stats.describe_with_ci(col("mean_sd"))},
        "regions": {},
    }
    for name in metrics.REGIONS:
        reg = {m: stats.describe(col(f"{name}_{m}")) for m in ("dice", "iou", "precision", "recall")}
        reg["dice"]["ci95"] = stats.bootstrap_ci(col(f"{name}_dice"))
        reg["hd95_mm"] = stats.describe(col(f"{name}_hd95_mm"))
        reg["abs_volume_error_ml"] = stats.describe(
            [abs(r[f"{name}_pred_ml"] - r[f"{name}_ref_ml"]) for r in rows])
        reg["n_ref_empty"] = int(sum(r[f"{name}_ref_ml"] == 0 for r in rows))
        if official and name in metrics.HECS:
            reg["dice_official"] = stats.describe_with_ci(col(f"{name}_dice_official"))
            reg["surface_dice_official"] = stats.describe_with_ci(col(f"{name}_sd_official"))
        out["regions"][name] = reg
    if official:
        out["dice_crosscheck_max_abs_diff"] = max(
            abs(r[f"{h}_dice"] - r[f"{h}_dice_official"]) for r in rows for h in metrics.HECS)
    return out


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--pred-dir", type=Path, required=True)
    ap.add_argument("--segs-root", type=Path, required=True)
    ap.add_argument("--split", type=Path, required=True)
    ap.add_argument("--role", default="test", choices=("test", "val"))
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--np", type=int, default=1, help="worker processes")
    ap.add_argument("--no-official", action="store_true", help="local smoke test only")
    args = ap.parse_args(argv)

    roles = json.loads(args.split.read_text(encoding="utf-8"))["roles"]
    cases = sorted(c for c, r in roles.items() if r == args.role)
    refs = manifest.find_case_files(args.segs_root)
    missing_pred = [c for c in cases if not (args.pred_dir / f"{c}.nii.gz").exists()]
    missing_ref = [c for c in cases if len(refs.get(c, [])) != 1]
    if missing_pred or missing_ref:
        print(f"STOP: missing predictions {missing_pred}; missing/duplicate references {missing_ref}")
        return 1
    official = not args.no_official
    if official:
        from project3d.eval import official as off

        off.check_config()
    jobs = [(c, args.pred_dir / f"{c}.nii.gz", refs[c][0], official) for c in cases]
    if args.np > 1:
        with Pool(args.np) as pool:
            rows = pool.map(score_case, jobs)
    else:
        rows = [score_case(j) for j in jobs]

    args.out_dir.mkdir(parents=True, exist_ok=True)
    with open(args.out_dir / "per_case.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    summary = {"spec": "SPEC 004", "role": args.role, "pred_dir": str(args.pred_dir),
               "split_fingerprint_sha256": json.loads(args.split.read_text())["fingerprint_sha256"],
               **summarize(rows, official)}
    if official:
        summary["kits23_evaluator"] = off.official_version()
    (args.out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    p = summary["primary"]
    print(f"{args.role}: n={len(rows)}  mean Dice {p['mean_dice'].get('mean', float('nan')):.4f} "
          f"CI {p['mean_dice']['ci95']}  mean surface Dice "
          f"{p['mean_surface_dice'].get('mean', float('nan')):.4f}")
    if official and summary["dice_crosscheck_max_abs_diff"] > CROSSCHECK_TOL:
        print(f"STOP: our Dice differs from the official Dice by "
              f"{summary['dice_crosscheck_max_abs_diff']:.2e}; check orientation/geometry")
        return 1
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/eval/case_analysis.py
"""SPEC 005 / E-005: performance by case characteristics, plus figures.

Example::

    python scripts/eval/case_analysis.py --per-case eval/per_case.csv \
        --case-features spec002/case_features.csv --images-root <images> \
        --segs-root <segmentations> --pred-dir pred_pp --out-dir analysis

Writes analysis.json, analysis.md and figures/ (box plots; best, median and
worst overlays). No GPU needed.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import manifest  # noqa: E402
from project3d.eval import analysis, figures  # noqa: E402


def read_csv(path: Path) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--per-case", type=Path, required=True)
    ap.add_argument("--case-features", type=Path, required=True)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--images-root", type=Path, help="needed for overlay figures")
    ap.add_argument("--segs-root", type=Path)
    ap.add_argument("--pred-dir", type=Path)
    args = ap.parse_args(argv)

    rows = analysis.join_rows(read_csv(args.per_case), read_csv(args.case_features))
    result = analysis.analyse(rows)
    args.out_dir.mkdir(parents=True, exist_ok=True)
    figdir = args.out_dir / "figures"
    made = [figures.boxplot_by_group(rows, "size_bin", "tumor_dice", analysis.SIZE_ORDER,
                                     figdir / "tumor_dice_by_size_bin.png"),
            figures.boxplot_by_group(rows, "cohort", "mean_dice", ["kits21", "kits23_new"],
                                     figdir / "mean_dice_by_cohort.png")]
    examples = analysis.pick_examples(rows)
    result["examples"] = examples
    if args.images_root and args.segs_root and args.pred_dir:
        import nibabel as nib

        imgs = manifest.find_case_files(args.images_root)
        segs = manifest.find_case_files(args.segs_root)
        score = {r["case_id"]: r["mean_dice"] for r in rows}
        for kind, cid in examples.items():
            ref_img = nib.load(str(segs[cid][0]))
            spacing = tuple(float(z) for z in ref_img.header.get_zooms()[:3])
            made.append(figures.overlay_figure(
                np.asanyarray(nib.load(str(imgs[cid][0])).dataobj, dtype=np.float32),
                np.asanyarray(ref_img.dataobj).astype(np.uint8),
                np.asanyarray(nib.load(str(args.pred_dir / f"{cid}.nii.gz")).dataobj).astype(np.uint8),
                spacing, f"{kind}: {cid}  mean Dice {score[cid]:.3f}",
                figdir / f"overlay_{kind}_{cid}.png"))
    result["figures"] = [p.name for p in made]
    (args.out_dir / "analysis.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    (args.out_dir / "analysis.md").write_text(analysis.to_markdown(result), encoding="utf-8")
    print(f"analysis: {len(rows)} cases, {len(made)} figures -> {args.out_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Step 1 - install nnU-Net (same pinned version as training) and the official evaluator
import importlib.metadata as md
def sh(cmd):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
sh([sys.executable, "-m", "pip", "install", "-q", "nnunetv2==2.8.1", "nibabel",
    "git+https://github.com/neheller/kits23"])
print("nnunetv2", md.version("nnunetv2"), "| torch", md.version("torch"), "| kits23", md.version("kits23"))

In [ ]:
# Step 2 - find the inputs: KiTS23 folders, trained model, locked split
sys.path[:0] = [f"{WORK}/src", f"{WORK}/scripts/data", f"{WORK}/scripts/eval"]
import prepare_kits23
from project3d.data import manifest
INPUT = Path("/kaggle/input")
IMAGES, SEGS = prepare_kits23.detect_roots(INPUT, min_files=489)
ckpts = sorted(INPUT.rglob("fold_0/checkpoint_final.pth"))
assert len(ckpts) == 1, f"need exactly one fold_0/checkpoint_final.pth, found {ckpts}"
FOLD = ckpts[0].parent
TRAINER = FOLD.parent  # e.g. .../nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres
for name in ("plans.json", "dataset.json"):
    assert (TRAINER / name).exists(), f"missing {TRAINER / name}"
splits = [p for p in INPUT.rglob("split.json") if "roles" in json.loads(p.read_text())]
assert len(splits) == 1, f"need exactly one SPEC 002 split.json, found {splits}"
SPLIT = splits[0]
FEATURES = SPLIT.parent / "case_features.csv"
assert FEATURES.exists(), FEATURES
split_record = json.loads(SPLIT.read_text())
roles = split_record["roles"]
TEST = sorted(c for c, r in roles.items() if r == "test")
VAL = sorted(c for c, r in roles.items() if r == "val")
print("images:", IMAGES, "| segmentations:", SEGS)
print("model:", TRAINER)
print("split:", SPLIT, "| fingerprint", split_record["fingerprint_sha256"][:16], "...")
print(f"{len(TEST)} test cases, {len(VAL)} validation cases")
for var in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):  # silences nnU-Net warnings
    os.environ.setdefault(var, f"/tmp/{var}")
OUT = Path("/kaggle/working/spec004")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 3 - predict the test cases (GPU). Test images are linked with nnU-Net's
# _0000 channel suffix; nothing is copied.
DISABLE_TTA = False  # True: about 8x faster, usually slightly lower Dice
images = manifest.find_case_files(IMAGES)
IMGS_TS = Path("/tmp/imagesTs")
IMGS_TS.mkdir(exist_ok=True)
for c in TEST:
    src = images[c][0]
    assert src.name.endswith(".nii.gz"), src
    dst = IMGS_TS / f"{c}_0000.nii.gz"
    if not dst.exists():
        dst.symlink_to(src)
t = time.time()
cmd = ["nnUNetv2_predict_from_modelfolder", "-i", IMGS_TS, "-o", OUT / "pred_raw",
       "-m", TRAINER, "-f", "0", "-chk", "checkpoint_final.pth"]
if DISABLE_TTA:
    cmd.append("--disable_tta")
sh(cmd)
predict_hours = (time.time() - t) / 3600
print(f"prediction took {predict_hours:.2f} h")

In [ ]:
# Step 4 - choose post-processing on the VALIDATION predictions only, apply it to test.
# nnU-Net writes postprocessing.pkl next to its input, so the validation folder is
# copied somewhere writable first.
VAL_SRC = FOLD / "validation"
val_found = sorted(p.name[:-7] for p in VAL_SRC.glob("case_*.nii.gz")) if VAL_SRC.is_dir() else []
pp_applied = val_found == VAL
if pp_applied:
    VAL_PRED, VAL_REF = Path("/tmp/val_pred"), Path("/tmp/val_ref")
    shutil.copytree(VAL_SRC, VAL_PRED, dirs_exist_ok=True)
    VAL_REF.mkdir(exist_ok=True)
    segs = manifest.find_case_files(SEGS)
    for c in VAL:
        if not (VAL_REF / f"{c}.nii.gz").exists():
            (VAL_REF / f"{c}.nii.gz").symlink_to(segs[c][0])
    common = ["-plans_json", TRAINER / "plans.json", "-dataset_json", TRAINER / "dataset.json", "-np", "4"]
    sh(["nnUNetv2_determine_postprocessing", "-i", VAL_PRED, "-ref", VAL_REF, *common])
    PKL = VAL_PRED / "postprocessing.pkl"
    sh(["nnUNetv2_apply_postprocessing", "-i", OUT / "pred_raw", "-o", OUT / "pred_pp",
        "-pp_pkl_file", PKL, *common])
    shutil.copy(PKL, OUT / "postprocessing.pkl")
    import pickle
    print("post-processing chosen on validation:", pickle.load(open(PKL, "rb")))
else:
    print(f"fold_0/validation has {len(val_found)} of {len(VAL)} expected cases: "
          "post-processing SKIPPED, the primary result uses the raw predictions.")
    shutil.copytree(OUT / "pred_raw", OUT / "pred_pp", dirs_exist_ok=True)

In [ ]:
# Step 5 - score both prediction sets with the official KiTS23 evaluator (CPU).
# evaluate.py stops if a prediction is missing or misaligned, or if our Dice and
# the official Dice disagree (a sign of an orientation bug).
for tag in ("pp", "raw"):
    sh([sys.executable, f"{WORK}/scripts/eval/evaluate.py", "--pred-dir", OUT / f"pred_{tag}",
        "--segs-root", SEGS, "--split", SPLIT, "--role", "test",
        "--out-dir", OUT / f"eval_{tag}", "--np", "4"])

In [ ]:
# Step 6 - SPEC 005: performance by case characteristics, figures
sh([sys.executable, f"{WORK}/scripts/eval/case_analysis.py",
    "--per-case", OUT / "eval_pp" / "per_case.csv", "--case-features", FEATURES,
    "--images-root", IMAGES, "--segs-root", SEGS, "--pred-dir", OUT / "pred_pp",
    "--out-dir", OUT / "analysis"])
from IPython.display import Image, Markdown, display
display(Markdown((OUT / "analysis" / "analysis.md").read_text()))
for png in sorted((OUT / "analysis" / "figures").glob("*.png")):
    display(Image(str(png)))

In [ ]:
# Step 7 - run record (copy the printed JSON back to Claude)
import hashlib, torch
def primary(tag):
    s = json.loads((OUT / f"eval_{tag}" / "summary.json").read_text())
    return {k: {"mean": v.get("mean"), "ci95": v.get("ci95")} for k, v in s["primary"].items()}, s
pp, summary = primary("pp")
raw, _ = primary("raw")
ckpt = FOLD / "checkpoint_final.pth"
record = {
    "spec": "SPEC 004", "nnunetv2": md.version("nnunetv2"), "torch": md.version("torch"),
    "kits23_evaluator": summary.get("kits23_evaluator"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "trainer_dir": TRAINER.name, "checkpoint_mb": round(ckpt.stat().st_size / 1e6, 1),
    "split_fingerprint_sha256": split_record["fingerprint_sha256"],
    "n_test_cases": summary["n_cases"], "disable_tta": DISABLE_TTA,
    "postprocessing_applied": pp_applied,
    "dice_crosscheck_max_abs_diff": summary.get("dice_crosscheck_max_abs_diff"),
    "primary_postprocessed": pp, "primary_raw": raw,
    "predict_hours": round(predict_hours, 2), "total_hours": round((time.time() - T0) / 3600, 2),
}
json.dump(record, open(OUT / "run_record.json", "w"), indent=2)
print(json.dumps(record, indent=1))
print("DONE. Save the Output as a private dataset (e.g. kits23-spec004-results).")